# ARGUS Phase 2.5: Attack Session Scoring & Detection Evaluation

Cross-references `redteam.txt` with session parquets, tokenizes attack sessions,
scores them, and compares against normal baselines.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, hashlib, bisect, csv, json, re, time
from collections import defaultdict

DATA_ROOT = Path("/kaggle/input/datasets/nightingale21/argus-tokenized-58day-verified/data")
SESSIONS_DIR = DATA_ROOT / "sessions"
TOKENIZED_ROOT = DATA_ROOT / "tokenized"
VOCAB_PATH = DATA_ROOT / "vocab.json"
VAL_MANIFEST = TOKENIZED_ROOT / "sessions_val.pt"

REDTEAM_PATH = Path("/kaggle/input/datasets/nightingale21/attacker/redteam.txt")

REPO_URL = "https://github.com/NIghtIngale340/ARGUS"
REPO_DIR = Path("/kaggle/working/argus-log-intelligence-platform")
REFRESH_REPO = True

EVAL_CHECKPOINT_DIR = Path("/kaggle/working/argus_mlm_eval_check")
CHECKPOINT_DIR = Path("/kaggle/working/argus_mlm_checkpoints")

SCORE_BATCH_SIZE = 128
SCORE_NUM_WORKERS = 2
MAX_SEQ_LEN = 16

ATTACK_OUT_DIR = Path("/kaggle/working/attack_sessions")
ATTACK_SCORES_CSV = Path("/kaggle/working/argus_attack_scores.csv")
NORMAL_SCORES_CSV = Path("/kaggle/working/argus_val_scores_sample.csv")
COMPARISON_REPORT = Path("/kaggle/working/argus_attack_comparison.json")


In [ ]:
def run_stream(command, cwd=None, env=None):
    print("$", " ".join(str(p) for p in command), flush=True)
    proc = subprocess.Popen(
        [str(p) for p in command], cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit {rc})")

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"free disk: {free / 1024**3:.1f} GB")


In [ ]:
if REFRESH_REPO and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    run_stream(["git", "clone", REPO_URL, str(REPO_DIR)], cwd="/kaggle/working")
else:
    run_stream(["git", "pull", "--ff-only"], cwd=REPO_DIR)

run_stream([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.37.0", "pyarrow>=14.0.0", "tqdm>=4.67.1", "pyyaml>=6.0",
    "scikit-learn",
], cwd=REPO_DIR)
print("Repo ready:", REPO_DIR)


## Validate Data Paths

In [ ]:
for p in [REDTEAM_PATH, VOCAB_PATH, VAL_MANIFEST, SESSIONS_DIR]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  {status}: {p}")
    if not p.exists():
        raise FileNotFoundError(p)

parquet_files = sorted(SESSIONS_DIR.glob("day_*.parquet"))
print(f"  Session parquets: {len(parquet_files)} files")


## Select Best Checkpoint

In [ ]:
eval_ckpts = sorted(EVAL_CHECKPOINT_DIR.glob("checkpoint_step_*.pt")) if EVAL_CHECKPOINT_DIR.exists() else []
train_ckpts = sorted(CHECKPOINT_DIR.glob("checkpoint_step_*.pt")) if CHECKPOINT_DIR.exists() else []

if eval_ckpts:
    BEST_CHECKPOINT = eval_ckpts[-1]
elif train_ckpts:
    BEST_CHECKPOINT = train_ckpts[-1]
else:
    raise FileNotFoundError("No checkpoint found. Upload/unzip your checkpoint first.")

print("Using checkpoint:", BEST_CHECKPOINT)


## Step 1: Parse redteam.txt & Find Attack Sessions

In [ ]:
import pyarrow.parquet as pq

def hash_id(raw_id):
    n = raw_id.strip() if raw_id else ""
    if not n:
        return "UNKNOWN"
    return hashlib.sha256(n.encode("utf-8")).hexdigest()[:8]

redteam_entries = []
with open(REDTEAM_PATH) as f:
    for line in f:
        parts = line.strip().split(",")
        if len(parts) != 4:
            continue
        try:
            redteam_entries.append({"time": int(parts[0]), "user": parts[1], "src": parts[2], "dst": parts[3]})
        except ValueError:
            continue

print(f"Parsed {len(redteam_entries):,} red-team events")

rt_lookup = defaultdict(list)
for e in redteam_entries:
    rt_lookup[(hash_id(e["user"]), hash_id(e["src"]))].append(e["time"])
for k in rt_lookup:
    rt_lookup[k].sort()

print(f"Unique attack (user, host) keys: {len(rt_lookup)}")
all_rt_times = [e["time"] for e in redteam_entries]
print(f"Red-team day range: {min(all_rt_times)//86400+1} to {max(all_rt_times)//86400+1}")


In [ ]:
attack_sessions = []
attack_labels = []
total_scanned = 0

for pq_path in parquet_files:
    match = re.search(r"day_(\d+)\.parquet$", pq_path.name)
    if not match:
        continue
    day_num = int(match.group(1))
    pf = pq.ParquetFile(pq_path)
    day_attacks = 0

    for batch in pf.iter_batches(batch_size=5000):
        for row in batch.to_pylist():
            total_scanned += 1
            uid = row.get("user_id", "")
            hid = row.get("host_id", "")
            key = (uid, hid)
            if key not in rt_lookup:
                continue
            st, et = row.get("start_ts", 0), row.get("end_ts", 0)
            left = bisect.bisect_left(rt_lookup[key], st)
            right = bisect.bisect_right(rt_lookup[key], et)
            if left >= right:
                continue
            attack_sessions.append(row)
            attack_labels.append({
                "idx": len(attack_labels), "day": day_num,
                "user_id": uid, "host_id": hid,
                "start_ts": st, "end_ts": et,
                "rt_events": right - left,
                "event_count": len(row.get("events", [])),
            })
            day_attacks += 1

    if day_attacks > 0:
        print(f"  day_{day_num:02d}: {day_attacks} attack session(s)")

print(f"\nTotal scanned: {total_scanned:,}")
print(f"Attack sessions found: {len(attack_sessions):,}")

if not attack_sessions:
    raise RuntimeError("No attack sessions found! Check hash matching.")


## Step 2: Tokenize Attack Sessions

In [ ]:
sys.path.insert(0, str(REPO_DIR))
from src.parsing.log_tokenizer import LogTokenizer
from collections.abc import Mapping

tokenizer = LogTokenizer(vocab_path=VOCAB_PATH, max_len=MAX_SEQ_LEN)
print(f"Vocab: {len(tokenizer.vocab)} tokens, max_len={MAX_SEQ_LEN}")

ATTACK_OUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = str(ATTACK_OUT_DIR / "attack_sessions.pt")

def coerce_events(events):
    if not isinstance(events, list):
        if hasattr(events, 'tolist'):
            events = events.tolist()
        elif isinstance(events, tuple):
            events = list(events)
        else:
            return []
    return [dict(e) for e in events if isinstance(e, Mapping)]

def session_gen():
    for row in attack_sessions:
        r = dict(row)
        r["events"] = coerce_events(r.get("events", []))
        yield r

stats = tokenizer.save_tokenized_sessions_pt_chunked_with_stats(
    sessions=session_gen(),
    output_path=manifest_path,
    chunk_size=5000,
    token_id_dtype=torch.int16,
    attention_mask_dtype=torch.bool,
)
print(f"Tokenized: {stats.session_count} sessions, {stats.chunk_count} chunks")

labels_path = ATTACK_OUT_DIR / "attack_session_labels.csv"
with open(labels_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(attack_labels[0].keys()))
    w.writeheader()
    w.writerows(attack_labels)
print(f"Labels: {labels_path}")


## Step 3: Score Attack Sessions

In [ ]:
attack_manifest = str(ATTACK_OUT_DIR / "attack_sessions.pt")

score_cmd = [
    sys.executable, "scripts/score_sessions.py",
    "--manifest", attack_manifest,
    "--checkpoint", str(BEST_CHECKPOINT),
    "--out", str(ATTACK_SCORES_CSV),
    "--batch-size", str(SCORE_BATCH_SIZE),
    "--num-workers", str(SCORE_NUM_WORKERS),
    "--log-every", "5000",
]
run_stream(score_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": str(REPO_DIR)})
print("Attack scores:", ATTACK_SCORES_CSV)


## Step 4: Score Normal Validation (if not already done)

In [ ]:
if NORMAL_SCORES_CSV.exists():
    print(f"Normal scores already exist: {NORMAL_SCORES_CSV}")
else:
    print("Scoring normal validation sessions (100 chunks)...")
    normal_cmd = [
        sys.executable, "scripts/score_sessions.py",
        "--manifest", str(VAL_MANIFEST),
        "--checkpoint", str(BEST_CHECKPOINT),
        "--out", str(NORMAL_SCORES_CSV),
        "--batch-size", str(SCORE_BATCH_SIZE),
        "--num-workers", str(SCORE_NUM_WORKERS),
        "--log-every", "50000",
        "--limit-chunks", "100",
    ]
    run_stream(normal_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": str(REPO_DIR)})


## Step 5: Compare Normal vs Attack

In [ ]:
import pandas as pd
import numpy as np

normal_df = pd.read_csv(NORMAL_SCORES_CSV)
attack_df = pd.read_csv(ATTACK_SCORES_CSV)
normal = normal_df["anomaly_score"].to_numpy()
attack = attack_df["anomaly_score"].to_numpy()

print(f"Normal sessions: {len(normal):,}")
print(f"Attack sessions: {len(attack):,}")
print(f"\nNormal: mean={normal.mean():.4f} std={normal.std():.4f}")
print(f"Attack: mean={attack.mean():.4f} std={attack.std():.4f}")

try:
    from sklearn.metrics import roc_auc_score
    labels = np.concatenate([np.zeros(len(normal)), np.ones(len(attack))])
    scores = np.concatenate([normal, attack])
    roc_auc = roc_auc_score(labels, scores)
except ImportError:
    roc_auc = 0.0

pooled_std = np.sqrt((normal.std()**2 + attack.std()**2) / 2)
cohens_d = (attack.mean() - normal.mean()) / pooled_std if pooled_std > 0 else 0
print(f"\nROC-AUC: {roc_auc:.6f}")
print(f"Cohen's d: {cohens_d:.4f}")

percentiles = {"p50": 50, "p90": 90, "p95": 95, "p99": 99, "p99.5": 99.5}
print(f"\n{'Thresh':<8} {'Value':>8} {'Recall':>8} {'FPR':>8} {'Prec':>8} {'F1':>8}")
print("-" * 50)

report = {"normal_count": len(normal), "attack_count": len(attack),
          "roc_auc": round(float(roc_auc), 6), "cohens_d": round(float(cohens_d), 4),
          "normal_mean": round(float(normal.mean()), 6),
          "attack_mean": round(float(attack.mean()), 6), "thresholds": {}}

for name, pct in percentiles.items():
    thr = np.percentile(normal, pct)
    tp = int((attack >= thr).sum())
    fp = int((normal >= thr).sum())
    fn = int((attack < thr).sum())
    tn = int((normal < thr).sum())
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    f1 = 2*prec*recall/(prec+recall) if (prec+recall) > 0 else 0
    print(f"{name:<8} {thr:>8.4f} {recall:>8.4f} {fpr:>8.4f} {prec:>8.4f} {f1:>8.4f}")
    report["thresholds"][name] = {"value": round(float(thr),6), "recall": round(float(recall),6),
        "precision": round(float(prec),6), "fpr": round(float(fpr),6), "f1": round(float(f1),6)}

COMPARISON_REPORT.write_text(json.dumps(report, indent=2))
print(f"\nReport saved: {COMPARISON_REPORT}")
if roc_auc >= 0.9: print("=> STRONG separation")
elif roc_auc >= 0.7: print("=> MODERATE separation")
else: print("=> WEAK separation")


## Archive Results

In [ ]:
ARCHIVE_DIR = Path("/kaggle/working/argus_phase25_attack_eval")
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

for a in [ATTACK_SCORES_CSV, NORMAL_SCORES_CSV, COMPARISON_REPORT,
          ATTACK_OUT_DIR / "attack_session_labels.csv"]:
    if a.exists():
        shutil.copy2(a, ARCHIVE_DIR / a.name)

archive = shutil.make_archive(str(ARCHIVE_DIR), "zip", root_dir=ARCHIVE_DIR)
print(f"Archive: {archive}")
print("Done! If ROC-AUC is strong, Phase 2 is COMPLETE.")
